# Settings and utilities

## Plot settings and data paths

In [1]:
import matplotlib.pyplot as plt
from google.colab import drive
from pathlib import Path

# Set common plot font sizes
plt.rcParams.update(
    {
        "font.size": 16,
        "axes.titlesize": 18,
        "axes.labelsize": 16,
        "xtick.labelsize": 14,
        "ytick.labelsize": 14,
        "legend.fontsize": 14,
        "figure.titlesize": 18,
    }
)

# Mount Google Drive and set the dataset root
# Change DATA_ROOT here if the practice folder is renamed or moved
drive.mount("/content/drive")
DATA_ROOT = Path("/content/drive/MyDrive/PHM Practice/Datasets")
PROJECT_ROOT = DATA_ROOT.parent

Mounted at /content/drive


## Functions

In [2]:
# Purpose: browse data files.
# Input: root dataset folder path.
# Output: interactive browser, scalar summaries, and indexed array previews.
# Used in: inspect MAT, CSV, and TXT data files.
def browse_data_files(root_dir):
    from pathlib import Path
    import numpy as np
    import pandas as pd
    import ipywidgets as widgets
    from ipyevents import Event
    from scipy.io import loadmat
    import h5py
    from IPython.display import display, clear_output

    # Initialize the browser and display controls
    root = Path(root_dir).resolve()
    current = root
    data = {}
    location = widgets.Label()
    file_label = widgets.Label()
    files = widgets.Select(rows=12, layout=widgets.Layout(width="650px"))
    open_button = widgets.Button(description="Open", icon="folder-open")
    up_button = widgets.Button(description="Up", icon="arrow-up")
    variables = widgets.Dropdown(description="Variable:")
    slices = widgets.HBox()
    result = widgets.Output()
    updating = False

    # Purpose: refresh.
    # Input: None; output: updated folder list.
    # Output: updated state or displayed result; no return value.
    def refresh():
        nonlocal updating
        updating = True
        try:
            location.value = str(current)
            entries = sorted(
                current.iterdir(), key=lambda p: (not p.is_dir(), p.name.lower())
            )
            files.options = [
                (f"[Folder] {p.name}" if p.is_dir() else p.name, str(p))
                for p in entries
                if p.is_dir() or p.suffix.lower() in (".mat", ".csv", ".txt")
            ]
            files.index = None
            up_button.disabled = current == root
        finally:
            updating = False

    # Purpose: indices.
    # Input: axis length; output: preview indices with an omission marker.
    # Output: returned list(range(size)) if size <= 6 else [0, 1, 2, 3, 4, None, size -
    # 1].
    def indices(size):
        return list(range(size)) if size <= 6 else [0, 1, 2, 3, 4, None, size - 1]

    # Purpose: format value.
    # Input: one table entry; output: display text.
    # Output: returned ''.join(value.astype(str).ravel()).
    def format_value(value):
        if isinstance(value, np.ndarray):
            # Unwrap single-element arrays to display their contents
            if value.size == 1:
                return format_value(value.item())
            if value.dtype.kind in ("U", "S"):
                return "".join(value.astype(str).ravel())
            return f"Array {value.shape}"
        if isinstance(value, np.void) and value.dtype.names:
            return "Fields: " + ", ".join(value.dtype.names)
        return f"{value:.8g}" if isinstance(value, (float, np.floating)) else str(value)

    # Purpose: show table.
    # Input: optional widget event; output: indexed preview table.
    # Output: updated state or displayed result; no return value.
    def show_table(_=None):
        with result:
            clear_output(wait=True)
            if variables.value is None:
                return
            value = np.asarray(data[variables.value])
            # Skip matlab structures unsupported by this viewer
            if value.dtype.names:
                print(f"{variables.value}: MATLAB structure")
                print("Structure preview is not supported by this viewer.")
                return
            print(f"{variables.value}: shape={value.shape}, dtype={value.dtype}")
            if value.size == 0:
                print("Empty array.")
                return
            if value.ndim == 0:
                print(format_value(value.item()))
                return
            if value.ndim > 2:
                positions = tuple(control.value - 1 for control in slices.children)
                value = value[positions]
                print(
                    "Selected positions:",
                    ", ".join(
                        f"Axis {i + 1} = {p + 1}" for i, p in enumerate(positions)
                    ),
                )
            if value.ndim == 1:
                value = value.reshape(-1, 1)

            # Preserve original row and column numbers in the compact preview
            rows = indices(value.shape[0])
            cols = indices(value.shape[1])
            table = pd.DataFrame(
                [
                    [
                        "…" if r is None or c is None else format_value(value[r, c])
                        for c in cols
                    ]
                    for r in rows
                ],
                index=["…" if r is None else r + 1 for r in rows],
                columns=["…" if c is None else c + 1 for c in cols],
            )
            table.index.name = "Row"
            table.columns.name = "Column"
            display(table)

    # Purpose: Select variable.
    # Input: optional widget event; output: slice controls and preview.
    # Output: updated state or displayed result; no return value.
    def select_variable(_=None):
        slices.children = ()
        if variables.value is None:
            return
        shape = np.asarray(data[variables.value]).shape
        if all(shape):
            slices.children = tuple(
                widgets.BoundedIntText(
                    value=1,
                    min=1,
                    max=size,
                    description=f"Axis {i + 1}:",
                    layout=widgets.Layout(width="180px"),
                )
                for i, size in enumerate(shape[:-2])
            )
        for control in slices.children:
            control.observe(show_table, names="value")
        show_table()

    # Purpose: open selected.
    # Input: Button or double-click event; output: folder listing or file preview.
    # Output: updated state or displayed result; no return value.
    def open_selected(_=None):
        nonlocal current, data
        if files.value is None:
            return
        path = Path(files.value)
        variables.options = []
        slices.children = ()
        file_label.value = ""
        with result:
            clear_output(wait=True)
            try:
                if path.is_dir():
                    current = path
                    refresh()
                    return

                # Load MAT variables or preserve every CSV row, including its header
                file_label.value = path.name
                if path.suffix.lower() == ".mat":
                    # Read matlab v7.3 arrays and restore their axis order
                    if h5py.is_hdf5(path):
                        with h5py.File(path, "r") as f:
                            data = {
                                name: np.asarray(value).T
                                for name, value in f.items()
                                if isinstance(value, h5py.Dataset)
                                and not name.startswith("#")
                            }
                    else:
                        loaded = loadmat(path)
                        data = {
                            name: value
                            for name, value in loaded.items()
                            if not name.startswith("__")
                        }
                elif path.suffix.lower() == ".csv":
                    data = {
                        "CSV": pd.read_csv(
                            path, header=None, low_memory=False
                        ).to_numpy()
                    }
                elif path.suffix.lower() == ".txt":
                    try:
                        table = pd.read_csv(path, sep=r"\s+", header=None)
                        data = {
                            "TXT": table.apply(pd.to_numeric, errors="raise").to_numpy()
                        }
                    except (pd.errors.ParserError, ValueError):
                        print(path.read_text(encoding="utf-8", errors="replace"))
                        return

                # Exclude matlab structures from the scalar summary
                scalars = [
                    (name, format_value(np.asarray(value).item()))
                    for name, value in data.items()
                    if np.asarray(value).size == 1 and not np.asarray(value).dtype.names
                ]
                if not data:
                    print("No variables found.")
                elif len(scalars) == len(data):
                    display(
                        pd.DataFrame(scalars, columns=["Variable", "Value"]).style.hide(
                            axis="index"
                        )
                    )
                else:
                    variables.options = list(data)
                    variables.value = max(
                        data, key=lambda name: np.asarray(data[name]).size
                    )
            except Exception as error:
                print(f"Cannot open: {error}")

    # Purpose: go up.
    # Input: Button event; output: updated folder list.
    # Output: updated state or displayed result; no return value.
    def go_up(_):
        nonlocal current
        if current != root:
            current = current.parent
            refresh()
            variables.options = []
            slices.children = ()
            file_label.value = ""
            with result:
                clear_output()

    # Purpose: on select.
    # Input: selection event; output: folder listing or file preview.
    # Output: updated state or displayed result; no return value.
    def on_select(change):
        if not updating and change["new"] is not None:
            open_selected()

    # Connect selection and button actions, then display the browser
    open_button.on_click(open_selected)
    up_button.on_click(go_up)
    variables.observe(select_variable, names="value")
    files.observe(on_select, names="value")
    refresh()
    display(
        widgets.VBox(
            [
                location,
                files,
                widgets.HBox([open_button, up_button]),
                file_label,
                variables,
                slices,
                result,
            ]
        )
    )

# Inspect MAT, CSV, TXT data files

In [3]:
# Browse MAT and CSV files under the dataset root
browse_data_files(DATA_ROOT)